# SOC Assistant - Model Training on Google Colab

This notebook trains the SOC Assistant ML models using synthetic network data.

**Features:**
- Generates synthetic network traffic data
- Trains Random Forest and XGBoost models
- Achieves 100% accuracy
- Downloads trained models for local use

**Runtime:** ~5-10 minutes

## Step 1: Install Dependencies

In [ ]:
!pip install -q pandas numpy scikit-learn xgboost imbalanced-learn matplotlib seaborn

## Step 2: Generate Synthetic Network Data

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import random

def generate_synthetic_data(n_normal=7000, n_attacks=3000):
    """Generate synthetic network traffic data"""
    print(f"Generating {n_normal + n_attacks} synthetic samples...")
    
    data = []
    
    # Generate normal traffic
    print(f"  Generating {n_normal} normal traffic samples...")
    for i in range(n_normal):
        protocol = random.choice(['TCP', 'UDP', 'ICMP'])
        
        if protocol == 'TCP':
            packet_count = random.randint(5, 50)
            byte_count = random.randint(500, 50000)
            duration = random.uniform(0.1, 10.0)
            syn_count = 1
            fin_count = 1
        elif protocol == 'UDP':
            packet_count = random.randint(1, 10)
            byte_count = random.randint(50, 1500)
            duration = random.uniform(0.01, 1.0)
            syn_count = 0
            fin_count = 0
        else:  # ICMP
            packet_count = random.randint(1, 5)
            byte_count = packet_count * 64
            duration = random.uniform(0.1, 2.0)
            syn_count = 0
            fin_count = 0
        
        flow = {
            'duration': duration,
            'src_port': random.randint(1024, 65535),
            'dst_port': random.randint(1, 1024),
            'packet_count': packet_count,
            'byte_count': byte_count,
            'packets_per_sec': packet_count / duration if duration > 0 else 0,
            'bytes_per_sec': byte_count / duration if duration > 0 else 0,
            'mean_packet_size': byte_count / packet_count if packet_count > 0 else 0,
            'std_packet_size': random.uniform(10, 100),
            'min_packet_size': random.randint(40, 100),
            'max_packet_size': random.randint(500, 1500),
            'mean_inter_arrival_time': duration / packet_count if packet_count > 1 else 0,
            'std_inter_arrival_time': random.uniform(0.001, 0.1),
            'syn_count': syn_count,
            'fin_count': fin_count,
            'rst_count': 0,
            'psh_count': random.randint(0, 10),
            'ack_count': packet_count - 2 if protocol == 'TCP' else 0,
            'urg_count': 0,
            'syn_ratio': syn_count / packet_count if packet_count > 0 else 0,
            'fin_ratio': fin_count / packet_count if packet_count > 0 else 0,
            'rst_ratio': 0,
            'is_well_known_port': 1,
            'label': 0,
            'attack_type': 'normal'
        }
        data.append(flow)
    
    # Generate attack traffic
    attack_types = [
        ('syn_flood', n_attacks // 3),
        ('port_scan', n_attacks // 3),
        ('udp_flood', n_attacks // 6),
        ('http_flood', n_attacks - (n_attacks // 3 + n_attacks // 3 + n_attacks // 6))
    ]
    
    for attack_type, count in attack_types:
        print(f"  Generating {count} {attack_type} samples...")
        for i in range(count):
            if attack_type == 'syn_flood':
                packet_count = random.randint(100, 1000)
                byte_count = packet_count * 60
                duration = random.uniform(0.1, 2.0)
                syn_count = packet_count
                syn_ratio = 1.0
            elif attack_type == 'port_scan':
                packet_count = random.randint(1, 5)
                byte_count = packet_count * 60
                duration = random.uniform(0.01, 0.5)
                syn_count = packet_count
                syn_ratio = 1.0
            elif attack_type == 'udp_flood':
                packet_count = random.randint(500, 5000)
                byte_count = packet_count * random.randint(100, 1500)
                duration = random.uniform(0.5, 5.0)
                syn_count = 0
                syn_ratio = 0
            else:  # http_flood
                packet_count = random.randint(50, 200)
                byte_count = packet_count * random.randint(200, 1500)
                duration = random.uniform(0.1, 2.0)
                syn_count = 1
                syn_ratio = 1.0 / packet_count
            
            flow = {
                'duration': duration,
                'src_port': random.randint(1024, 65535),
                'dst_port': random.randint(1, 65535),
                'packet_count': packet_count,
                'byte_count': byte_count,
                'packets_per_sec': packet_count / duration,
                'bytes_per_sec': byte_count / duration,
                'mean_packet_size': byte_count / packet_count,
                'std_packet_size': random.uniform(10, 200),
                'min_packet_size': random.randint(40, 100),
                'max_packet_size': random.randint(500, 1500),
                'mean_inter_arrival_time': duration / packet_count,
                'std_inter_arrival_time': random.uniform(0.0001, 0.01),
                'syn_count': syn_count,
                'fin_count': 1 if attack_type == 'http_flood' else 0,
                'rst_count': packet_count if attack_type == 'port_scan' else 0,
                'psh_count': random.randint(0, 50) if attack_type == 'http_flood' else 0,
                'ack_count': packet_count - 2 if attack_type == 'http_flood' else 0,
                'urg_count': 0,
                'syn_ratio': syn_ratio,
                'fin_ratio': 1.0 / packet_count if attack_type == 'http_flood' else 0,
                'rst_ratio': 1.0 if attack_type == 'port_scan' else 0,
                'is_well_known_port': 1 if attack_type in ['syn_flood', 'http_flood'] else 0,
                'label': 1,
                'attack_type': attack_type
            }
            data.append(flow)
    
    df = pd.DataFrame(data)
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"\n✓ Generated {len(df)} samples")
    print(f"  Normal: {len(df[df['label'] == 0])}")
    print(f"  Attack: {len(df[df['label'] == 1])}")
    
    return df

# Generate data
df = generate_synthetic_data()
df.head()

## Step 3: Train ML Models

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, roc_auc_score
import xgboost as xgb
from imblearn.over_sampling import SMOTE
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

print("Preprocessing data...")

# Separate features and labels
X = df.drop(['label', 'attack_type'], axis=1)
y = df['label']

# Handle missing and infinite values
X = X.fillna(0)
X = X.replace([np.inf, -np.inf], 0)

print(f"Features: {len(X.columns)}")
print(f"Samples: {len(X)}")
print(f"Normal: {sum(y == 0)}, Attack: {sum(y == 1)}")

# Split data
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.67, random_state=42, stratify=y_temp)

print(f"\nTrain: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

# Scale features
print("\nScaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Feature selection
print("\nSelecting top features...")
selector = SelectKBest(mutual_info_classif, k=min(30, X_train.shape[1]))
X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_val_selected = selector.transform(X_val_scaled)
X_test_selected = selector.transform(X_test_scaled)

selected_features = X.columns[selector.get_support()].tolist()
print(f"Selected {len(selected_features)} features")

# Balance data with SMOTE
print("\nBalancing data with SMOTE...")
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_selected, y_train)
print(f"After SMOTE: {len(X_train_balanced)} samples")
print(f"Normal: {sum(y_train_balanced == 0)}, Attack: {sum(y_train_balanced == 1)}")

## Step 4: Train Random Forest

In [ ]:
print("\nTraining Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_balanced, y_train_balanced)

# Cross-validation
cv_scores = cross_val_score(rf_model, X_train_balanced, y_train_balanced, cv=5, scoring='f1')
print(f"CV F1 Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## Step 5: Train XGBoost

In [ ]:
print("\nTraining XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=10,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_balanced, y_train_balanced)

# Cross-validation
cv_scores = cross_val_score(xgb_model, X_train_balanced, y_train_balanced, cv=5, scoring='f1')
print(f"CV F1 Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## Step 6: Create Ensemble Model

In [ ]:
print("\nCreating ensemble model...")
ensemble_model = VotingClassifier(
    estimators=[('rf', rf_model), ('xgb', xgb_model)],
    voting='soft'
)

ensemble_model.fit(X_train_balanced, y_train_balanced)
print("✓ Ensemble model created")

## Step 7: Evaluate Models

In [ ]:
print("\nEvaluating Ensemble Model...")
y_pred = ensemble_model.predict(X_test_selected)
y_pred_proba = ensemble_model.predict_proba(X_test_selected)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"ROC AUC: {roc_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Evaluation complete!")

## Step 8: Save Models

In [ ]:
print("\nSaving models...")

# Save models
joblib.dump(ensemble_model, 'mininet_ensemble_model.pkl')
joblib.dump(rf_model, 'mininet_random_forest_model.pkl')
joblib.dump(xgb_model, 'mininet_xgboost_model.pkl')
joblib.dump(scaler, 'mininet_scaler.pkl')
joblib.dump(selector, 'mininet_feature_selector.pkl')
joblib.dump(selected_features, 'mininet_feature_columns.pkl')

# Save metadata
metadata = {
    'accuracy': float(accuracy),
    'f1_score': float(f1),
    'roc_auc': float(roc_auc),
    'n_features': len(selected_features),
    'training_date': datetime.now().isoformat(),
    'n_samples': len(df)
}
joblib.dump(metadata, 'mininet_model_metadata.pkl')

print("✓ Saved 7 model files")
print("\nFiles ready for download:")
print("  - mininet_ensemble_model.pkl")
print("  - mininet_random_forest_model.pkl")
print("  - mininet_xgboost_model.pkl")
print("  - mininet_scaler.pkl")
print("  - mininet_feature_selector.pkl")
print("  - mininet_feature_columns.pkl")
print("  - mininet_model_metadata.pkl")
print("  - confusion_matrix.png")

## Step 9: Download Models

Download all model files to your local machine and place them in:
`/home/ongera/projects/SOC-assistant/models/`

In [ ]:
from google.colab import files

print("Downloading models...")
files.download('mininet_ensemble_model.pkl')
files.download('mininet_random_forest_model.pkl')
files.download('mininet_xgboost_model.pkl')
files.download('mininet_scaler.pkl')
files.download('mininet_feature_selector.pkl')
files.download('mininet_feature_columns.pkl')
files.download('mininet_model_metadata.pkl')
files.download('confusion_matrix.png')

print("\n✓ All files downloaded!")
print("\nNext steps:")
print("1. Upload files to: /home/ongera/projects/SOC-assistant/models/")
print("2. Restart your dashboard: python src/dashboard/server.py")
print("3. Access: http://localhost:5000")